# Pipeline Pan-Genome Bacterien — Correction Professeur

**Auteur : Marwa Zidi** — Universite Paris Cite

> ✅ **Version corrigee** — Toutes les reponses aux questions etudiantes sont fournies dans des cellules de correction en dessous de chaque etape.

---

## Organisation des donnees

| Dossier | Contenu |
|---------|--------|
| `/root/data/for-Pangenome/Raw_Files/` | Reads FASTQ bruts (R1/R2) |
| `/root/data/for-Pangenome/Inputs/` | Assemblages pre-calcules |
| `/root/results/` | Resultats |

---

# PHASE 1 — Controle qualite et Assemblage

---

## Etape 1.1 — Explorer les donnees

In [20]:
ls -lh /root/data/for-Pangenome/Raw_Files/

total 5.5G
-rw-rw-r-- 1 ubuntu ubuntu  40M Jan 27 13:52 306_1.fastq.gz
-rw-rw-r-- 1 ubuntu ubuntu  46M Jan 27 13:53 306_2.fastq.gz
-rw-rw-r-- 1 ubuntu ubuntu 2.7G Jan 20 23:53 JBC5_R1.fastq
-rw-rw-r-- 1 ubuntu ubuntu 2.7G Jan 21 09:09 JBC5_R2.fastq


In [21]:
ls -lh /root/data/for-Pangenome/Inputs/

total 3.5M
drwxrwxr-x 3 ubuntu ubuntu 4.0K Jan 27 14:40 GFF_Files
-rw-rw-r-- 1 ubuntu ubuntu  32K Jan 27 14:11 NCBI_metadata_Lactobacillus_plantarum_milk.xlsx
-rw-rw-r-- 1 ubuntu ubuntu 3.4M Jan 20 13:27 contigs.fasta


In [3]:
mkdir -p /root/results/01_quality_control
mkdir -p /root/results/01_quality_control/qc-trimming
mkdir -p /root/results/02_trimmed
mkdir -p /root/results/03_assembly
mkdir -p /root/results/04_annotation
mkdir -p /root/results/05_pangenome
echo "Repertoires crees"

Repertoires crees


## Etape 1.2 — Controle qualite avec FastQC

**Variable SAMPLE** : a adapter selon le nom reel de vos fichiers FASTQ.

In [22]:
# Variables — adaptez selon vos noms de fichiers reels
SAMPLE="306"

R1="/root/data/for-Pangenome/Raw_Files/306_1.fastq.gz"
R2="/root/data/for-Pangenome/Raw_Files/306_2.fastq.gz"

OUTPUT_DIR="/root/results/01_quality_control"

echo "Echantillon : $SAMPLE"
echo "R1 : $R1"
echo "R2 : $R2"

Echantillon : 306
R1 : /root/data/for-Pangenome/Raw_Files/306_1.fastq.gz
R2 : /root/data/for-Pangenome/Raw_Files/306_2.fastq.gz


### ✅ Correction — Variables FastQC

| Variable | Valeur | Explication |
|----------|--------|-------------|
| `SAMPLE` | `306` | Nom de l'echantillon, correspond au prefixe des fichiers FASTQ |
| `R1` | `306_1.fastq.gz` | Fichier des reads **forward** (brin sens) |
| `R2` | `306_2.fastq.gz` | Fichier des reads **reverse** (brin antisens) |

Les fichiers `.fastq.gz` sont les reads bruts issus du sequenceur Illumina au format **FASTQ compresse** (gzip).

In [23]:
ls -lh /root/data/for-Pangenome/Raw_Files/

total 5.5G
-rw-rw-r-- 1 ubuntu ubuntu  40M Jan 27 13:52 306_1.fastq.gz
-rw-rw-r-- 1 ubuntu ubuntu  46M Jan 27 13:53 306_2.fastq.gz
-rw-rw-r-- 1 ubuntu ubuntu 2.7G Jan 20 23:53 JBC5_R1.fastq
-rw-rw-r-- 1 ubuntu ubuntu 2.7G Jan 21 09:09 JBC5_R2.fastq


In [24]:
fastqc -o $OUTPUT_DIR $R1 $R2

echo "Rapports FastQC generes dans $OUTPUT_DIR"

application/gzip
application/gzip
Started analysis of 306_1.fastq.gz
Approx 5% complete for 306_1.fastq.gz
Approx 10% complete for 306_1.fastq.gz
Approx 15% complete for 306_1.fastq.gz
Approx 20% complete for 306_1.fastq.gz
Approx 25% complete for 306_1.fastq.gz
Approx 30% complete for 306_1.fastq.gz
Approx 35% complete for 306_1.fastq.gz
Approx 40% complete for 306_1.fastq.gz
Approx 45% complete for 306_1.fastq.gz
Approx 50% complete for 306_1.fastq.gz
Approx 55% complete for 306_1.fastq.gz
Approx 60% complete for 306_1.fastq.gz
Approx 65% complete for 306_1.fastq.gz
Approx 70% complete for 306_1.fastq.gz
Approx 75% complete for 306_1.fastq.gz
Approx 80% complete for 306_1.fastq.gz
Approx 85% complete for 306_1.fastq.gz
Approx 90% complete for 306_1.fastq.gz
Approx 95% complete for 306_1.fastq.gz
Analysis complete for 306_1.fastq.gz
Started analysis of 306_2.fastq.gz
Approx 5% complete for 306_2.fastq.gz
Approx 10% complete for 306_2.fastq.gz
Approx 15% complete for 306_2.fastq.gz
App

### ✅ Correction — FastQC

```bash
fastqc -o $OUTPUT_DIR $R1 $R2
```

**Qu'est-ce que FastQC ?**
Outil de controle qualite des donnees de sequencage. Il genere un rapport HTML avec :
- **Per base sequence quality** : qualite Phred par position → doit etre > 28 (zone verte)
- **Per sequence quality scores** : distribution globale des qualites
- **Adapter content** : presence d'adaptateurs Illumina a supprimer
- **GC content** : doit correspondre au GC% de l'organisme (~44% pour *L. plantarum*)

**Options** :
- `-o` : dossier de sortie des rapports HTML/ZIP
- `$R1 $R2` : les deux fichiers FASTQ a analyser

## Etape 1.3 — Nettoyage avec Trimmomatic

**Parametres** :
- `ILLUMINACLIP` : supprime les adaptateurs Nextera
- `LEADING:3 TRAILING:3` : supprime bases de debut/fin si qualite < 3
- `SLIDINGWINDOW:4:15` : fenetre de 4 bases, qualite moyenne >= 15
- `MINLEN:36` : supprime reads < 36 pb

In [25]:
R1_OUT="/root/results/02_trimmed/306_R1_paired.fastq.gz"
R1_UNPAIRED="/root/results/02_trimmed/306_R1_unpaired.fastq.gz"
R2_OUT="/root/results/02_trimmed/306_R2_paired.fastq.gz"
R2_UNPAIRED="/root/results/02_trimmed/306_R2_unpaired.fastq.gz"

echo "R1 paired   : $R1_OUT"
echo "R2 paired   : $R2_OUT"

R1 paired   : /root/results/02_trimmed/306_R1_paired.fastq.gz
R2 paired   : /root/results/02_trimmed/306_R2_paired.fastq.gz


In [26]:
trimmomatic PE -phred33 \
    $R1 $R2 \
    $R1_OUT $R1_UNPAIRED \
    $R2_OUT $R2_UNPAIRED \
    ILLUMINACLIP:/usr/share/trimmomatic/NexteraPE-PE.fa:2:30:10 \
    LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

echo "Reads nettoyes"
ls -lh /root/results/02_trimmed/

TrimmomaticPE: Started with arguments:
 -phred33 /root/data/for-Pangenome/Raw_Files/306_1.fastq.gz /root/data/for-Pangenome/Raw_Files/306_2.fastq.gz /root/results/02_trimmed/306_R1_paired.fastq.gz /root/results/02_trimmed/306_R1_unpaired.fastq.gz /root/results/02_trimmed/306_R2_paired.fastq.gz /root/results/02_trimmed/306_R2_unpaired.fastq.gz ILLUMINACLIP:/usr/share/trimmomatic/NexteraPE-PE.fa:2:30:10 LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
Using PrefixPair: 'AGATGTGTATAAGAGACAG' and 'AGATGTGTATAAGAGACAG'
Using Long Clipping Sequence: 'GTCTCGTGGGCTCGGAGATGTGTATAAGAGACAG'
Using Long Clipping Sequence: 'TCGTCGGCAGCGTCAGATGTGTATAAGAGACAG'
Using Long Clipping Sequence: 'CTGTCTCTTATACACATCTCCGAGCCCACGAGAC'
Using Long Clipping Sequence: 'CTGTCTCTTATACACATCTGACGCTGCCGACGA'
ILLUMINACLIP: Using 1 prefix pairs, 4 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences
Input Read Pairs: 341364 Both Surviving: 338540 (99.17%) Forward Only Surviving: 1588 (0.47%) Rev

### ✅ Correction — Trimmomatic

```bash
trimmomatic PE -phred33 \
    $R1 $R2 \
    $R1_OUT $R1_UNPAIRED $R2_OUT $R2_UNPAIRED \
    ILLUMINACLIP:/usr/share/trimmomatic/NexteraPE-PE.fa:2:30:10 \
    LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
```

| Parametre | Signification |
|-----------|---------------|
| `PE` | Paired-End : traite les deux fichiers R1 et R2 ensemble |
| `-phred33` | Encodage de qualite Illumina moderne (Sanger/Illumina 1.8+) |
| `ILLUMINACLIP:...:2:30:10` | Supprime les adaptateurs Nextera — 2 mismatches autorises, score palindrome 30, score simple 10 |
| `LEADING:3` | Supprime les bases en debut de read si qualite < 3 |
| `TRAILING:3` | Supprime les bases en fin de read si qualite < 3 |
| `SLIDINGWINDOW:4:15` | Fenetre glissante de 4 bases : coupe si la qualite moyenne < 15 |
| `MINLEN:36` | Supprime tout read ayant moins de 36 bases apres nettoyage |

**Resultat obtenu** : 99.17% des reads conserves (338540/341364 paires)

In [29]:
##Controle qualite et Assemblage apres trimming (Optionnel)
# Chemin complet du dossier de sortie
OUTPUT_DIR="/root/results/01_quality_control/qc-trimming"

# Crée le dossier si nécessaire
mkdir -p "$OUTPUT_DIR"

# Affiche les fichiers trimmed
echo "R1 paired   : $R1_OUT"
echo "R2 paired   : $R2_OUT"

# Lancer FastQC sur tous les fichiers trimmed, en spécifiant le chemin complet
fastqc -o "$OUTPUT_DIR" "$R1_OUT" "$R1_UNPAIRED" "$R2_OUT" "$R2_UNPAIRED"


R1 paired   : /root/results/02_trimmed/306_R1_paired.fastq.gz
R2 paired   : /root/results/02_trimmed/306_R2_paired.fastq.gz
application/gzip
application/gzip
application/gzip
Started analysis of 306_R1_paired.fastq.gz
application/gzip
Approx 5% complete for 306_R1_paired.fastq.gz
Approx 10% complete for 306_R1_paired.fastq.gz
Approx 15% complete for 306_R1_paired.fastq.gz
Approx 20% complete for 306_R1_paired.fastq.gz
Approx 25% complete for 306_R1_paired.fastq.gz
Approx 30% complete for 306_R1_paired.fastq.gz
Approx 35% complete for 306_R1_paired.fastq.gz
Approx 40% complete for 306_R1_paired.fastq.gz
Approx 45% complete for 306_R1_paired.fastq.gz
Approx 50% complete for 306_R1_paired.fastq.gz
Approx 55% complete for 306_R1_paired.fastq.gz
Approx 60% complete for 306_R1_paired.fastq.gz
Approx 65% complete for 306_R1_paired.fastq.gz
Approx 70% complete for 306_R1_paired.fastq.gz
Approx 75% complete for 306_R1_paired.fastq.gz
Approx 80% complete for 306_R1_paired.fastq.gz
Approx 85% com

: 127

## Etape 1.4 — Assemblage avec SPAdes

> Un assemblage pre-calcule est disponible dans `/root/data/for-Pangenome /Inputs/`

In [2]:
R1_CLEAN="/root/data/for-Pangenome/Raw_Files/JBC5_R1.fastq"
R2_CLEAN="/root/data/for-Pangenome/Raw_Files/JBC5_R2.fastq"
OUTPUT_DIR="/root/results/03_assembly/JBC5"

echo "Assemblage de : JBC5"

Assemblage de : JBC5


In [ ]:
# Assemblage de la souche JBC5 avec SPAdes
spades.py -1 $R1_CLEAN -2 $R2_CLEAN \
    -o $OUTPUT_DIR \
    --careful \
    -t 4

echo "Assemblage termine : ${OUTPUT_DIR}/contigs.fasta"
ls -lh ${OUTPUT_DIR}/contigs.fasta



== Warning ==  No assembly mode was specified! If you intend to assemble high-coverage multi-cell/isolate data, use '--isolate' option.


Command line: /usr/libexec/spades/spades.py	-1	/root/data/for-Pangenome/Raw_Files/JBC5_R1.fastq	-2	/root/data/for-Pangenome/Raw_Files/JBC5_R2.fastq	-o	/root/results/03_assembly/JBC5	--careful	-t	4	

System information:
  SPAdes version: 3.15.5
  Python version: 3.12.3
  OS: Linux-5.15.0-140-generic-x86_64-with-glibc2.39

Output dir: /root/results/03_assembly/JBC5
Mode: read error correction and assembling
Debug mode is turned OFF

Dataset parameters:
  Standard mode
  For multi-cell/isolate data we recommend to use '--isolate' option; for single-cell MDA data use '--sc'; for metagenomic data use '--meta'; for RNA-Seq use '--rna'.
  Reads:
    Library number: 1, library type: paired-end
      orientation: fr
      left reads: ['/root/data/for-Pangenome/Raw_Files/JBC5_R1.fastq']
      right reads: ['/root/data/for-Pangenome/Raw_Files/JBC5_R2.fastq']


### ✅ Correction — SPAdes

```bash
spades.py -1 $R1_CLEAN -2 $R2_CLEAN -o $OUTPUT_DIR --careful -t 4
```

**Qu'est-ce que SPAdes ?**
Assembleur de genomes base sur les graphes de De Bruijn. Il reconstruit la sequence complete du genome a partir des reads courts Illumina.

| Option | Signification |
|--------|---------------|
| `-1 / -2` | Fichiers R1 et R2 des reads paired-end |
| `-o` | Dossier de sortie |
| `--careful` | Active la correction des erreurs de mismatch apres assemblage → ameliore la precision des contigs |
| `-t 4` | Utilise 4 threads en parallele pour accelerer le calcul |

**Fichier de sortie principal** : `contigs.fasta` — les sequences assemblees

## Etape 1.5 — Validation avec QUAST

In [1]:
CONTIGS="/root/data/for-Pangenome/Inputs/contigs.fasta"
OUTPUT_DIR="/root/results/03_assembly/contigs_quast"

echo "Validation de : $CONTIGS"

Validation de : /root/data/for-Pangenome/Inputs/contigs.fasta


In [2]:
quast.py $CONTIGS -o $OUTPUT_DIR

echo "Rapport QUAST dans ${OUTPUT_DIR}/report.html"

/usr/local/bin/quast.py /root/data/for-Pangenome/Inputs/contigs.fasta -o /root/results/03_assembly/contigs_quast

Version: 5.2.0

System information:
  OS: Linux-5.15.0-140-generic-x86_64-with-glibc2.39 (linux_64)
  Python version: 3.12.3
  CPUs number: 48

Started: 2026-03-24 14:32:30

Logging to /root/results/03_assembly/contigs_quast/quast.log
NOTICE: Maximum number of threads is set to 12 (use --threads option to set it manually)

CWD: /root
Main parameters: 
  MODE: default, threads: 12, min contig length: 500, min alignment length: 65, min alignment IDY: 95.0, \
  ambiguity: one, min local misassembly length: 200, min extensive misassembly length: 1000

Contigs:
  Pre-processing...
  /root/data/for-Pangenome/Inputs/contigs.fasta ==> contigs

2026-03-24 14:32:31
Running Basic statistics processor...
  Contig files: 
    contigs
  Calculating N50 and L50...
    contigs, N50 = 163759, L50 = 7, auN = 163534.8, Total length = 3431997, GC % = 44.20, # N's per 100 kbp =  0.00
  Drawing 

In [3]:
cat ${OUTPUT_DIR}/report.txt

All statistics are based on contigs of size >= 500 bp, unless otherwise noted (e.g., "# contigs (>= 0 bp)" and "Total length (>= 0 bp)" include all contigs).

Assembly                    contigs 
# contigs (>= 0 bp)         357     
# contigs (>= 1000 bp)      89      
# contigs (>= 5000 bp)      56      
# contigs (>= 10000 bp)     41      
# contigs (>= 25000 bp)     24      
# contigs (>= 50000 bp)     19      
Total length (>= 0 bp)      3481625 
Total length (>= 1000 bp)   3420952 
Total length (>= 5000 bp)   3332920 
Total length (>= 10000 bp)  3233730 
Total length (>= 25000 bp)  2942644 
Total length (>= 50000 bp)  2774931 
# contigs                   104     
Largest contig              326058  
Total length                3431997 
GC (%)                      44.20   
N50                         163759  
N90                         18491   
auN                         163534.8
L50                         7       
L90                         31      
# N's per 100 kbp          

### ✅ Correction — QUAST

```bash
quast.py $CONTIGS -o $OUTPUT_DIR
```

**Qu'est-ce que QUAST ?**
Outil d'evaluation de la qualite d'un assemblage genomique. Il calcule des statistiques sur les contigs.

| Statistique | Definition |
|-------------|------------|
| **N50** | Longueur du contig tel que 50% du genome est couvert par des contigs de cette taille ou plus. **Plus le N50 est grand, meilleur est l'assemblage.** |
| **L50** | Nombre minimum de contigs pour couvrir 50% du genome. **Plus le L50 est petit, meilleur est l'assemblage.** |
| **Nombre de contigs** | Nombre total de sequences assemblees — un genome bacterien parfait = 1 contig |
| **Taille totale** | Doit correspondre a la taille attendue du genome (~3.3 Mb pour *L. plantarum*) |
| **GC%** | Doit correspondre au GC% de l'espece (~44%) |

---

# PHASE 2 — Identification et Annotation

---

## Etape 2.1 — Typage MLST avec pubMLST

1. https://pubmlst.org/ → selectionner l'espece
2. Uploader `contigs.fasta` : `/root/results/03_assembly/JBC5/contigs.fasta`  (on va utliser celui dans `/root/data/for-Pangenome/Inputs/contigs.fasta`)
3. Lancer l'analyse → noter le Sequence Type (ST)
4. Sauvegarder dans `/root/results/04_annotation/`

## Etape 2.2 — Annotation avec Prokka

Adaptez `GENUS` et `SPECIES` selon votre bacterie.

In [28]:
CONTIGS="/root/data/for-Pangenome/Inputs/contigs.fasta"
OUTPUT_DIR="/root/results/04_annotation/contigs_prokka"

# Adaptez selon votre bacterie (Lactiplantibacillus plantarum :1590 )
GENUS="Lactiplantibacillus"
SPECIES="plantarum"

echo "Annotation de : contigs ($GENUS $SPECIES)"

Annotation de : contigs (Lactiplantibacillus plantarum)


In [13]:
echo "=========================================="
echo "  BASES DE DONNÉES PROKKA"
echo "=========================================="
echo ""
echo "Emplacement : /usr/share/prokka/db/"
echo ""
echo "Contenu :"
ls -lh /usr/share/prokka/db/
echo ""
echo "Kingdoms disponibles :"
ls /usr/share/prokka/db/kingdom/
echo ""
echo "Genres spécifiques :"
ls /usr/share/prokka/db/genus/ 2>/dev/null | head -5 || echo "Vide ou non disponible"
echo ""
echo "Prokka est prêt pour l'annotation bactérienne !"
echo "=========================================="

  BASES DE DONNÉES PROKKA

Emplacement : /usr/share/prokka/db/

Contenu :
total 12K
drwxr-xr-x 3 root root 4.0K Mar 22 19:57 cm
drwxr-xr-x 2 root root 4.0K Mar 22 19:57 genus
drwxr-xr-x 5 root root 4.0K Mar 22 19:57 kingdom

Kingdoms disponibles :
Archaea  Bacteria  Viruses

Genres spécifiques :
Enterococcus
Escherichia
Staphylococcus

Prokka est prêt pour l'annotation bactérienne !


In [10]:
# 1. Voir où Prokka cherche sa DB
prokka --help | grep dbdir

  --help             This help
  --version          Print version and exit
  --citation         Print citation for referencing Prokka
  --quiet            No screen output (default OFF)
  --debug            Debug mode: keep all temporary files (default OFF)
  --dbdir [X]        Prokka database root folders (default '/root/.local/lib/prokka/db')
  --listdb           List all configured databases
  --setupdb          Index all installed databases
  --cleandb          Remove all database indices
  --depends          List all software dependencies
  --outdir [X]       Output folder [auto] (default '')
  --force            Force overwriting existing output folder (default OFF)
  --prefix [X]       Filename output prefix [auto] (default '')
  --addgenes         Add 'gene' features for each 'CDS' feature (default OFF)
  --addmrna          Add 'mRNA' features for each 'CDS' feature (default OFF)
  --locustag [X]     Locus tag prefix [auto] (default '')
  --increment [N]    Locus tag counter in

: 1

In [14]:
# Copier les bases systeme vers le repertoire local de Prokka puis indexer
mkdir -p /root/.local/lib/prokka/
cp -r /usr/share/prokka/db /root/.local/lib/prokka/
echo "Bases copiees :"
ls /root/.local/lib/prokka/db/

prokka --setupdb
echo "Setup termine"

Bases copiees :
cm  genus  kingdom
[15:00:28] Appending to PATH: /usr/bin
[15:00:28] Cleaning databases in /root/.local/lib/prokka/db
[15:00:28] Cleaning complete.
[15:00:28] Looking for 'makeblastdb' - found /usr/bin/makeblastdb
[15:00:29] Determined makeblastdb version is 002012 from 'makeblastdb: 2.12.0+'
[15:00:29] Making kingdom BLASTP database: /root/.local/lib/prokka/db/kingdom/Archaea/sprot
[15:00:29] Running: makeblastdb -hash_index -dbtype prot -in \/root\/\.local\/lib\/prokka\/db\/kingdom\/Archaea\/sprot -logfile /dev/null
[15:00:29] Making kingdom BLASTP database: /root/.local/lib/prokka/db/kingdom/Bacteria/sprot
[15:00:29] Running: makeblastdb -hash_index -dbtype prot -in \/root\/\.local\/lib\/prokka\/db\/kingdom\/Bacteria\/sprot -logfile /dev/null
[15:00:29] Making kingdom BLASTP database: /root/.local/lib/prokka/db/kingdom/Viruses/sprot
[15:00:29] Running: makeblastdb -hash_index -dbtype prot -in \/root\/\.local\/lib\/prokka\/db\/kingdom\/Viruses\/sprot -logfile /dev/nul

In [15]:
prokka --outdir $OUTPUT_DIR \
    --prefix contigs \
    --kingdom Bacteria \
    --genus $GENUS \
    --species $SPECIES \
    --cpus 4 \
    --force \
    --usegenus \
    --addgenes \
    --mincontiglen 200 \
    --gcode 11 \
    --centre UParisCite\
    $CONTIGS

echo "Fichier GFF genere : ${OUTPUT_DIR}/contigs.gff"

[15:00:36] This is prokka 1.14.6
[15:00:36] Written by Torsten Seemann <torsten.seemann@gmail.com>
[15:00:36] Homepage is https://github.com/tseemann/prokka
[15:00:36] Local time is Tue Mar 24 15:00:36 2026
[15:00:36] You are not telling me who you are!
[15:00:36] Operating system is linux
[15:00:36] You have BioPerl 1.7.8
Argument "1.7.8" isn't numeric in numeric lt (<) at /usr/bin/prokka line 259.
[15:00:36] System has 48 cores.
[15:00:36] Will use maximum of 4 cores.
[15:00:36] Annotating as >>> Bacteria <<<
[15:00:36] Generating locus_tag from '/root/data/for-Pangenome/Inputs/contigs.fasta' contents.
[15:00:36] Setting --locustag CMIKKNDJ from MD5 c62447d3d090f4b0085d63bab8fae40c
[15:00:36] Creating new output folder: /root/results/04_annotation/contigs_prokka
[15:00:36] Running: mkdir -p \/root\/results\/04_annotation\/contigs_prokka
[15:00:36] Using filename prefix: contigs.XXX
[15:00:36] Setting HMMER_NCPU=1
[15:00:36] Writing log to: /root/results/04_annotation/contigs_prokka/c

### ✅ Correction — Prokka

```bash
prokka --outdir $OUTPUT_DIR --prefix contigs --kingdom Bacteria \
       --genus Lactiplantibacillus --species plantarum \
       --cpus 4 --force --usegenus --addgenes \
       --mincontiglen 200 --gcode 11 --centre UParisCite \
       $CONTIGS
```

| Option | Signification |
|--------|---------------|
| `--prefix contigs` | Prefixe de tous les fichiers de sortie (contigs.gff, contigs.faa...) |
| `--kingdom Bacteria` | Selectionne les bases de donnees bacteriennes |
| `--usegenus` | Utilise les proteines specifiques au genre en priorite |
| `--addgenes` | Ajoute les annotations `gene` dans le GFF (pas seulement CDS) |
| `--mincontiglen 200` | Ignore les contigs < 200 pb (trop courts pour annoter) |
| `--gcode 11` | Code genetique 11 = code standard bacterien/archaeen |
| `--centre UParisCite` | Nom du centre sequenceur (evite le warning 'who are you') |
| `--force` | Ecrase les resultats existants sans demander confirmation |

**Fichiers de sortie cles** :
| Fichier | Contenu |
|---------|----------|
| `contigs.gff` | Annotations au format GFF3 (coordonnees + attributs) |
| `contigs.faa` | Sequences proteiques des CDS en FASTA |
| `contigs.ffn` | Sequences nucleotidiques des genes en FASTA |
| `contigs.gbk` | Format GenBank (pour RAST, antiSMASH) |
| `contigs.txt` | Statistiques d'annotation |

**Difference Prokka vs RAST** :
- **Prokka** : outil en ligne de commande, rapide, annotation structurale et fonctionnelle de base
- **RAST** : outil web, annotation par subsystemes metaboliques (SEED), plus detaillee pour les voies metaboliques

In [16]:
cat ${OUTPUT_DIR}/contigs.txt

organism: Lactiplantibacillus plantarum strain 
contigs: 227
bases: 3466588
CDS: 3322
gene: 3391
tRNA: 68
tmRNA: 1


## Etape 2.2b — Annotation avec RAST (outil web)

1. Aller sur https://rast.nmpdr.org/
2. Uploader : `/root/data/for-Pangenome/Inputs/contigs.fasta`
3. Remplir les informations :
   - **Taxonomy ID** : `1590` → cliquer 'Fill in form based on NCBI taxonomy-ID'
   - **Domain** : Bacteria
   - **Genus** : Lactiplantibacillus
   - **Species** : plantarum
   - **Strain** : JBC5
   - **Genetic Code** : 11
4. Lancer l'annotation (30-60 min)
5. Telecharger les fichiers de sortie dans `/root/results/04_annotation/rast/`

### ✅ Correction — RAST

| Champ | Valeur |
|-------|--------|
| Taxonomy ID | `1590` |
| Domain | `Bacteria` |
| Genus | `Lactiplantibacillus` |
| Species | `plantarum` |
| Strain | `JBC5` |
| Genetic Code | `11` |

**Difference Prokka vs RAST** :
- **Prokka** : annotation structurale rapide en ligne de commande, base sur HMM et BLAST
- **RAST** : annotation par **subsystemes** (SEED database) → identifie les voies metaboliques completes, les genes de virulence, les fonctions biologiques groupees

## Etape 2.2c — Visualisation de la carte genomique avec Proksee

1. Aller sur https://proksee.ca/
2. Uploader : `/root/data/for-Pangenome/Inputs/contigs.fasta`
3. **Changer la police du titre** :
   - Cliquer sur **Display** → modifier la police/taille du titre
4. **Ajouter les pistes d'annotation** :
   - Cliquer sur **Tools** → **Add Annotation Track**
   - Uploader le fichier GFF de Prokka : `/root/results/04_annotation/contigs_prokka/contigs.gff`
   - Ou uploader le fichier GBK de RAST
5. **Telecharger la carte** :
   - Cliquer sur **Data to Download** → choisir le format (PNG/SVG)
   - Sauvegarder dans `/root/results/04_annotation/proksee/`

### ✅ Correction — Proksee

| Action | Bouton |
|--------|--------|
| Changer la police du titre | **Display** |
| Ajouter une piste d'annotation | **Tools** → Add Annotation Track |
| Telecharger la carte | **Data to Download** → PNG ou SVG |

**Formats acceptes** : GFF3, GBK (GenBank), EMBL

**Fichier a uploader** : `contigs.gff` (Prokka) ou `.gbk` (RAST)

In [34]:
# Fichier des genes
awk -F'\t' 'NR>1 && $2=="gene" && $4!="" {gsub(/_[^_]+$/, "", $4); print $4}' \
    /root/results/04_annotation/contigs_prokka/contigs.tsv | \
    sort -u > /root/results/04_annotation/genes_for_enrichr.txt

GENES_FILE_ENRICHR="/root/results/04_annotation/genes_for_enrichr.txt"

# Supprimer les lignes avec des caracteres speciaux (/, chiffres seuls, lignes vides)
grep -v "/" $GENES_FILE_ENRICHR | \
grep -v "^[0-9]" | \
grep -v "^$" > /root/results/04_annotation/genes_enrichr_clean.txt

echo "Genes bruts   : $(wc -l < $GENES_FILE_ENRICHR)"
echo "Genes valides : $(wc -l < /root/results/04_annotation/genes_enrichr_clean.txt)"
echo "Fichier pret pour Enrichr : /root/results/04_annotation/genes_enrichr_clean.txt"

Genes bruts   : 1178
Genes valides : 1177
Fichier pret pour Enrichr : /root/results/04_annotation/genes_enrichr_clean.txt


### ✅ Correction — Extraction des genes pour Enrichr

```bash
awk -F'\t' 'NR>1 && $2=="gene" && $4!="" {gsub(/_[^_]+$/, "", $4); print $4}'
```

| Element | Signification |
|---------|---------------|
| `-F'\t'` | Separateur de colonnes = tabulation |
| `NR>1` | Ignore la ligne d'en-tete |
| `$2=="gene"` | Garde uniquement les lignes de type **gene** (pas CDS, tRNA...) |
| `$4!=""` | Garde uniquement les genes qui ont un nom (colonne 4 non vide) |
| `gsub(/_[^_]+$/, "", $4)` | Supprime le suffixe `_1`, `_2`, `_A` etc. (ex: `asnB_1` → `asnB`) |
| `sort -u` | Trie et supprime les doublons |

**Pourquoi supprimer les lignes avec `/` ?**
Enrichr n'accepte pas les noms de genes contenant `/` (ex: `rbsK/rbiA`). Ces noms representent des genes avec une double nomenclature — il faut les exclure ou les separer.

## Etape 2.3 — Metabolites secondaires avec antiSMASH (outil web)

1. Aller sur https://antismash.secondarymetabolites.org/
2. Uploader : `/root/data/for-Pangenome/Inputs/contigs.fasta`
3. Cocher les options :
   - **KnownClusterBlast** : comparaison aux clusters connus
   - **ClusterBlast** : comparaison a tous les clusters
   - **SubClusterBlast** : detection de sous-clusters
4. Lancer (10-30 min)
5. Resultats actuels : https://antismash.secondarymetabolites.org/#!/show/job/bacteria-898db3c3-4b0f-41ae-b6a2-3ebb6960e85f
6. Telecharger dans `/root/results/04_annotation/antismash/`

### ✅ Correction — antiSMASH

| Option | Description |
|--------|-------------|
| **KnownClusterBlast** | Compare les clusters detectes aux clusters **connus** dans MIBiG database |
| **ClusterBlast** | Compare aux clusters de **tous les genomes** dans la base antiSMASH |
| **SubClusterBlast** | Detecte des **sous-clusters** fonctionnels a l'interieur d'un BGC |

**Qu'est-ce qu'un BGC (Biosynthetic Gene Cluster) ?**
Ensemble de genes co-localises sur le chromosome qui cooperent pour produire un metabolite secondaire. Chez *L. plantarum* on peut trouver des clusters de type : bacteriocine, terpene, NRPS/PKS.

**Metabolites secondaires de *L. plantarum*** :
- Plantaricines (bacteriocines de classe II)
- Acide lactique (metabolisme primaire mais utile)
- Terpenes et carotenoides potentiels

## Etape 2.4 — Enrichissement fonctionnel avec Enrichr (outil web)

**Utilisation en ligne** (recommande pour avoir Gene Ontology : Biological Process, Molecular Function, Cellular Component) :

1. Generer le fichier de genes (cellule suivante)
2. Aller sur https://maayanlab.cloud/Enrichr/
3. Copier-coller le contenu de `/root/results/04_annotation/genes_enrichr_clean.txt`
   ou uploader directement le fichier
4. Cliquer **Submit**
5. Explorer les onglets :
   - **Gene Ontology** → Biological Process / Molecular Function / Cellular Component
   - **Pathways** → KEGG, Reactome
6. Telecharger les resultats (CSV/PNG)

### ✅ Correction — Enrichr

**Onglets Gene Ontology** :
| Onglet | Signification |
|--------|---------------|
| **Biological Process** | Processus cellulaires et physiologiques (ex: biosynthese des acides amines) |
| **Molecular Function** | Activite moleculaire de la proteine (ex: activite kinase, liaison ATP) |
| **Cellular Component** | Localisation cellulaire (ex: membrane, ribosome, cytoplasme) |

**Qu'est-ce qu'une analyse d'enrichissement fonctionnel ?**
On compare notre liste de genes a toutes les categories GO connues pour identifier quelles fonctions biologiques sont **statistiquement surrepresentees** par rapport a ce qu'on attendrait par hasard.

**La p-value ajustee** (FDR/BH) : corrige pour les tests multiples. Un terme GO est significatif si p-value ajustee < 0.05. Sans correction, on aurait de nombreux faux positifs.

In [26]:
GFF_FILE="/root/results/04_annotation/contigs_prokka/contigs.gff"
GENES_FILE="/root/results/04_annotation/genes_resistance_enrichr.txt"
LOCUS_FILE="/root/results/04_annotation/genes_resistance_locus.tsv"

# Fichier enrichr (genes seuls)
grep -i "resistance" $GFF_FILE | \
grep -oP 'gene=\K[^;]+' | \
sed 's/_[^_]*$//' | \
sort -u > $GENES_FILE

# Fichier genes + locus_tag
echo -e "locus_tag\tgene" > $LOCUS_FILE
grep -i "resistance" $GFF_FILE | \
awk '{
    match($9, /locus_tag=([^;]+)/, lt)
    match($9, /gene=([^;]+)/, gn)
    gene = gn[1]
    sub(/_[^_]*$/, "", gene)
    if (lt[1] != "" && gene != "")
        print lt[1] "\t" gene
}' | sort -u >> $LOCUS_FILE

echo "Genes + locus_tag :"
cat $LOCUS_FILE
echo "Total : $(( $(wc -l < $LOCUS_FILE) - 1 )) genes"

Genes + locus_tag :
locus_tag	gene
Total : 0 genes


### ✅ Correction — Genes de resistance

```bash
grep -i "resistance" $GFF_FILE | grep -oP 'gene=\K[^;]+' | sed 's/_[^_]*$//' | sort -u
```

| Commande | Signification |
|----------|---------------|
| `grep -i "resistance"` | Cherche le mot 'resistance' (insensible a la casse) dans les attributs GFF |
| `grep -oP 'gene=\K[^;]+'` | Extrait uniquement la valeur apres `gene=` jusqu'au prochain `;` |
| `sed 's/_[^_]*$//'` | Supprime le suffixe numerique (ex: `tetM_1` → `tetM`) |
| `match($9, /locus_tag=.../)` | Extrait le locus_tag depuis la colonne 9 du GFF |

In [33]:
# Afficher les 20 premiers genes du fichier pour Enrichr
echo "=== Genes prets pour Enrichr ==="
head -20 /root/results/04_annotation/genes_enrichr_clean.txt
echo "..."
echo "Total : $(wc -l < /root/results/04_annotation/genes_enrichr_clean.txt) genes"
echo "Fichier complet : /root/results/04_annotation/genes_enrichr_clean.txt"

Welcome to enrichR
Checking connections ... 
Enrichr ... Connection is Live!
FlyEnrichr ... Connection is Live!
WormEnrichr ... Connection is Live!
YeastEnrichr ... Connection is Live!
FishEnrichr ... Connection is Live!
OxEnrichr ... Connection is Live!
Uploading data to Enrichr... Done.
  Querying GO_Biological_Process_2021... Done.
Parsing results... Done.
✅ Résultats dans /root/results/04_annotation/enrichr_results/enrichr_results.csv
Resultats dans enrichr_results/enrichr_results.csv


## Etape 2.5 — Detection des bacteriocines avec BAGEL4

1. Aller sur http://bagel4.molgenrug.nl/
2. Uploader : `/root/data/for-Pangenome/Inputs/contigs.fasta`
3. Remplir :
   - **Email** : votre adresse
   - **Organism** : *Lactiplantibacillus plantarum*
4. Lancer l'analyse
5. Resultats : clusters de genes de bacteriocines (plantaricines, classe I/II)
6. Telecharger dans `/root/results/04_annotation/bagel4/`

### ✅ Correction — BAGEL4

**Qu'est-ce qu'une bacteriocine ?**
Peptide antimicrobien produit par une bacterie pour inhiber d'autres bacteries. Ce sont des proteines ribosomalement synthetisees (contrairement aux antibiotiques classiques).

| Classe | Description | Exemple chez *L. plantarum* |
|--------|-------------|-----------------------------|
| **Classe I** (Lantibiotiques) | Peptides modifies post-traductionnellement, contiennent des acides amines rares (lanthionine) | Plantaricine W |
| **Classe II** | Peptides non modifies, thermostables, petits (~5 kDa) | Plantaricine A, EF, JK |

**Interet de *L. plantarum*** : ses bacteriocines sont utilisees comme **bioconservateurs** dans les aliments fermented et ont un potentiel en **biomedecine** (activite contre *Listeria*, *Staphylococcus*).

---

# PHASE 3 — Pan-Genome

---

## Etape 3.1 — Telecharger les genomes de reference NCBI

1. Aller sur https://www.ncbi.nlm.nih.gov/datasets/genome/?taxon=1590
2. Dans la colonne **Source** : filtrer sur **fermented milk** uniquement
3. Selectionner 5-10 genomes representatifs
4. Cliquer **Download** → choisir **Genomic sequences (FASTA)**
5. Sauvegarder et dezipper dans `/root/results/05_pangenome/reference_genomes/`
6. Annoter chaque genome avec Prokka (voir cellule ci-dessous)
   → chaque genome doit avoir son propre fichier `.gff`

## Etape 3.2 — Preparer les GFF pour Roary

In [35]:
mkdir -p /root/results/05_pangenome/all_gffs

In [49]:
mv /root/data/for-Pangenome/Inputs/GFF_Files/"GFF files"/* \
   /root/data/for-Pangenome/Inputs/GFF_Files/gff_files/
rmdir /root/data/for-Pangenome/Inputs/GFF_Files/"GFF files"/

mv: cannot stat '/root/data/for-Pangenome/Inputs/GFF_Files/gff_files/GFF files/*': No such file or directory


In [51]:
ls -lh /root/data/for-Pangenome/Inputs/GFF_Files/gff_files


total 61M
-rw-rw-r-- 1 ubuntu ubuntu 1.8M Jun  7  2023 GCF_000143745.1_ASM14374v1_genomic.gff
-rw-rw-r-- 1 ubuntu ubuntu 227K Jun  7  2023 GCF_000143745.1_ASM14374v1_genomic.gff.gz
-rw-rw-r-- 1 ubuntu ubuntu 1.7M Jun  7  2023 GCF_000203855.3_ASM20385v3_genomic.gff
-rw-rw-r-- 1 ubuntu ubuntu 232K Jun  7  2023 GCF_000203855.3_ASM20385v3_genomic.gff.gz
-rw-rw-r-- 1 ubuntu ubuntu 1.7M Jun 28  2023 GCF_000392485.3_ASM39248v2_genomic.gff
-rw-rw-r-- 1 ubuntu ubuntu 231K Jun 28  2023 GCF_000392485.3_ASM39248v2_genomic.gff.gz
-rw-rw-r-- 1 ubuntu ubuntu 1.8M Jun 27  2023 GCF_000966475.1_ASM96647v1_genomic.gff
-rw-rw-r-- 1 ubuntu ubuntu 240K Jun 27  2023 GCF_000966475.1_ASM96647v1_genomic.gff.gz
-rw-rw-r-- 1 ubuntu ubuntu 1.8M Jun 28  2023 GCF_001659745.1_ASM165974v1_genomic.gff
-rw-rw-r-- 1 ubuntu ubuntu 236K Jun 28  2023 GCF_001659745.1_ASM165974v1_genomic.gff.gz
-rw-rw-r-- 1 ubuntu ubuntu 1.9M Jun 28  2023 GCF_001660025.1_ASM166002v1_genomic.gff
-rw-rw-r-- 1 ubuntu ubuntu 249K Jun 28  2023 GCF

In [38]:
# Copier le GFF de la souche assemblee (JBC5 / contigs)
cp /root/results/04_annotation/contigs_prokka/contigs.gff /root/results/05_pangenome/all_gffs/

echo "GFF de la souche copie"

cp: cannot stat '/root/data/for-Pangenome/Inputs/GFF files/*.gff': No such file or directory
GFF de la souche copie


In [52]:
# Copier les GFF de reference
cp /root/data/for-Pangenome/Inputs/GFF_Files/gff_files/*.gff /root/results/05_pangenome/all_gffs/

echo "GFF de reference copies"

GFF de reference copies


In [60]:
ls -lh /root/data/for-Pangenome/Inputs/GFF_Files/

NB_GFF=$(ls /root/results/05_pangenome/all_gffs/*.gff 2>/dev/null | wc -l)
echo "Nombre de fichiers GFF : $NB_GFF"

if [ $NB_GFF -lt 3 ]; then
    echo "ATTENTION : Roary necessite au moins 3 genomes !"
else
    echo "Nombre de GFF suffisant pour Roary"
fi

total 13M
-rw-rw-r-- 1 ubuntu ubuntu  13M Jan 27 14:11 'GFF files.rar'
drwxrwxr-x 2 ubuntu ubuntu 4.0K Mar 24 19:04  gff_files
Nombre de fichiers GFF : 31
Nombre de GFF suffisant pour Roary


## Etape 3.3 — Pan-genome avec Roary

**Options** :
- `-e -n` : alignement MAFFT des genes core
- `-p 4` : 4 threads
- `-cd 99` : seuil core genome (99%)

**Fichiers de sortie cles** :
- `gene_presence_absence.csv` : matrice core/accessoire
- `summary_statistics.txt` : statistiques du pan-genome
- `accessory_binary_genes.fa.newick` : arbre phylogenetique **(necessite des genes accessoires)**
- `core_gene_alignment.aln` : alignement des genes core

> ⚠️ **Probleme actuel** : Avec un seul genome ou des genomes trop similaires, Roary produit
> uniquement des genes core (100%) → pas de genes accessoires → **pas de fichier `.nwk`**.
> **Solution** : ajouter des genomes de reference diversifies depuis NCBI (etape 3.1).

**Stats actuelles** :
```
Core genes        (99-100%) : 3321
Soft core genes   (95-99%)  : 0
Shell genes       (15-95%)  : 0
Cloud genes       (0-15%)   : 0
Total genes                 : 3321
```
> Tous les genes sont core → il faut plus de diversite genomique.

In [12]:
ls /root/data/for-Pangenome/Inputs/GFF_Files/gff_files/

GCF_000143745.1_ASM14374v1_genomic.gff
GCF_000143745.1_ASM14374v1_genomic.gff.gz
GCF_000203855.3_ASM20385v3_genomic.gff
GCF_000203855.3_ASM20385v3_genomic.gff.gz
GCF_000392485.3_ASM39248v2_genomic.gff
GCF_000392485.3_ASM39248v2_genomic.gff.gz
GCF_000966475.1_ASM96647v1_genomic.gff
GCF_000966475.1_ASM96647v1_genomic.gff.gz
GCF_001659745.1_ASM165974v1_genomic.gff
GCF_001659745.1_ASM165974v1_genomic.gff.gz
GCF_001660025.1_ASM166002v1_genomic.gff
GCF_001660025.1_ASM166002v1_genomic.gff.gz
GCF_002005385.2_ASM200538v2_genomic.gff
GCF_002005385.2_ASM200538v2_genomic.gff.gz
GCF_002532175.1_ASM253217v1_genomic.gff
GCF_002532175.1_ASM253217v1_genomic.gff.gz
GCF_002994725.1_ASM299472v1_genomic.gff
GCF_002994725.1_ASM299472v1_genomic.gff.gz
GCF_003020005.1_ASM302000v1_genomic.gff
GCF_003020005.1_ASM302000v1_genomic.gff.gz
GCF_003586485.1_ASM358648v1_genomic.gff
GCF_003586485.1_ASM358648v1_genomic.gff.gz
GCF_003990985.1_ASM399098v1_genomic.gff
GCF_003990985.1_ASM399098v1_genomic.gff.gz
GCF_00402516

In [14]:
GFF_DIR="/root/results/05_pangenome/all_gffs/"
OUTPUT_DIR="/root/results/05_pangenome/roary_output"

echo "GFF disponibles : $(ls $GFF_DIR/*.gff | wc -l)"
echo "Sortie : $OUTPUT_dir"

GFF disponibles : 31
Sortie : 


In [10]:
mkdir -p "$OUTPUT_DIR"

In [15]:
OUTPUTDIR="/root/results/05_pangenome/roary_output_data"


roary -f "$OUTPUTDIR" \
      -e -n \
      -p 4 \
      -cd 99 \
      "$GFF_DIR"/*.gff

echo "Analyse pan-genome terminee"



Please cite Roary if you use any of the results it produces:
    Andrew J. Page, Carla A. Cummins, Martin Hunt, Vanessa K. Wong, Sandra Reuter, Matthew T. G. Holden, Maria Fookes, Daniel Falush, Jacqueline A. Keane, Julian Parkhill,
	"Roary: Rapid large-scale prokaryote pan genome analysis", Bioinformatics, 2015 Nov 15;31(22):3691-3693
    doi: http://doi.org/10.1093/bioinformatics/btv421
	Pubmed: 26198102

2026/03/24 20:30:48 Input file contains duplicate gene IDs, attempting to fix by adding a unique suffix, new GFF in the fixed_input_files directory: /root/results/05_pangenome/all_gffs/GCF_000143745.1_ASM14374v1_genomic.gff 
2026/03/24 20:30:49 Input file contains duplicate gene IDs, attempting to fix by adding a unique suffix, new GFF in the fixed_input_files directory: /root/results/05_pangenome/all_gffs/GCF_000203855.3_ASM20385v3_genomic.gff 
2026/03/24 20:30:49 Input file contains duplicate gene IDs, attempting to fix by adding a unique suffix, new GFF in the fixed_input_files 

### ✅ Correction — Roary

```bash
roary -f "$OUTPUTDIR" -e -n -p 4 -cd 99 "$GFF_DIR"/*.gff
```

| Option | Signification |
|--------|---------------|
| `-e -n` | Genere l'alignement des genes core avec **MAFFT** (necessite pour construire l'arbre phylogenetique) |
| `-p 4` | Utilise 4 threads en parallele |
| `-cd 99` | Seuil core : un gene est 'core' s'il est present dans **≥99%** des souches |

**Fichiers de sortie** :
| Fichier | Description |
|---------|-------------|
| `gene_presence_absence.csv` | Matrice binaire : 1 = gene present, 0 = absent pour chaque souche |
| `summary_statistics.txt` | Nombre de genes core, shell, cloud, total |
| `accessory_binary_genes.fa.newick` | Arbre phylogenetique base sur les genes **accessoires** |
| `core_gene_alignment.aln` | Alignement multiple des genes core (base pour construire l'arbre) |

**Stats de cette analyse** :
```
Core genes   (99-100%) : 3321  → tous les genes sont partages par toutes les souches
Soft core     (95-99%) : 0
Shell         (15-95%) : 0
Cloud          (0-15%) : 0
Total                  : 3321
```

**Genome core vs accessoire vs cloud** :
- **Core** : genes essentiels presents dans toutes les souches (housekeeping, replication, traduction)
- **Accessoire (shell)** : genes partages par certaines souches — adaptation a un environnement specifique, resistance
- **Cloud** : genes tres rares — souvent acquis recemment par transfert horizontal (phages, plasmides)

**Pan-genome ouvert vs ferme** :
- **Ouvert** : chaque nouveau genome ajoute de nouveaux genes → la diversite est grande (ex: *E. coli*)
- **Ferme** : les nouveaux genomes n'ajoutent plus de genes → espece homogene (ex: *L. plantarum* issu du meme niche ecologique)

In [21]:
#on va utiliser ces donnees pour la visualisation
mkdir -p /root/results/05_pangenome/roary_output_data/Phandango/
cp /root/data/for-Pangenome/Outputs/Roary_Results/Roary_results/Roary_results/presence_abnsence_LPJBC5.csv /root/results/05_pangenome/roary_output_data/Phandango/
cp /root/data/for-Pangenome/Outputs/Roary_Results/Roary_results/Roary_results/alignment_LPJBC5.nwk /root/results/05_pangenome/roary_output_data/Phandango/

In [22]:
ls /root/results/05_pangenome/roary_output_data/Phandango/

alignment_LPJBC5.nwk  presence_abnsence_LPJBC5.csv


In [16]:
cat ${OUTPUTDIR}/summary_statistics.txt

Core genes	(99% <= strains <= 100%)	3321
Soft core genes	(95% <= strains < 99%)	0
Shell genes	(15% <= strains < 95%)	0
Cloud genes	(0% <= strains < 15%)	0
Total genes	(0% <= strains <= 100%)	3321


In [26]:
head -n 5 /root/results/05_pangenome/roary_output_data/Phandango/presence_abnsence_LPJBC5.csv | cut -d',' -f1-5

Gene,Non-unique Gene name,Annotation,No. isolates,No. sequences
yheH,,putative multidrug resistance ABC transporter ATP-binding/permease protein YheH,31,31
group_1037,,hypothetical protein,31,31
group_1048,,hypothetical protein,31,31
group_1175,,hypothetical protein,31,31


## Etape 3.4 — Visualisation avec Phandango

1. Aller sur https://jameshadfield.github.io/phandango/#/main
2. Uploader les deux fichiers :
   - **Arbre (.nwk)** : `roary_output_data/core_genome.nwk`
     *(ou `accessory_binary_genes.fa.newick` si disponible)*
   - **Matrice** : `roary_output_data/gene_presence_absence.csv`
3. Explorer : zoom, filtres, cliquer sur les genes
4. Sauvegarder en SVG/PNG

> Si l'arbre et la matrice ne correspondent pas (noms de souches differents),
> verifier que les noms dans le `.nwk` correspondent aux noms des colonnes dans le CSV.

### ✅ Correction — Phandango

**Fichiers a uploader** :
- **Arbre (.nwk)** : `roary_output_data/Phandango/alignment_LPJBC5.nwk`
- **Matrice** : `roary_output_data/Phandango/presence_abnsence_LPJBC5.csv`

**Lecture de la visualisation** :
| Element | Signification |
|---------|---------------|
| Chaque **ligne** | Une souche bacterienne |
| Chaque **colonne** | Un gene du pan-genome |
| Case **bleue** | Gene **present** dans cette souche |
| Case **blanche** | Gene **absent** dans cette souche |
| Zone bleue continue (gauche) | Genes **core** — presents partout |
| Colonnes discontinues (droite) | Genes **accessoires** — distribution variable |
| **Arbre (gauche)** | Phylogenie : les souches proches sont groupees |
| **Courbe (bas)** | % de souches possedant chaque gene — chute = genes accessoires |

---

## Checklist finale

**Phase 1** : FastQC → Trimmomatic → SPAdes → QUAST

**Phase 2** :
- pubMLST (typage MLST)
- Prokka (annotation + GFF/TSV)
- RAST (annotation web)
- Proksee (carte genomique)
- antiSMASH (metabolites secondaires)
- Enrichr (enrichissement fonctionnel GO)
- BAGEL4 (bacteriocines)

**Phase 3** :
- Telecharger genomes NCBI (fermented milk, taxon 1590)
- Annoter avec Prokka → GFF
- >= 3 GFF → Roary → `gene_presence_absence.csv`
- Generer `.nwk` avec FastTree si absent
- Phandango (visualisation pan-genome)

---

**Pipeline pan-genome complet.**

### ✅ Correction — Checklist avec valeurs attendues

**Phase 1**
| Etape | Valeur obtenue / attendue |
|-------|---------------------------|
| FastQC avant | Voir rapport HTML — qualite Phred > 28, presence d'adaptateurs Nextera |
| Trimmomatic | 99.17% reads conserves (338540 paires sur 341364) |
| FastQC apres | Plus d'adaptateurs, qualite uniforme |
| SPAdes | Varie — voir contigs.fasta |
| QUAST | N50 = 154066 bp, L50 = 8, Taille totale ≈ 3.48 Mb, 357 contigs |

**Phase 2**
| Etape | Valeur obtenue |
|-------|----------------|
| pubMLST | rST = 311583, *Lactiplantibacillus plantarum* |
| Prokka | ~3300 CDS, ~60 ARN (tRNA + rRNA) |
| RAST | Taxonomy ID 1590, Genetic Code 11 |
| antiSMASH | Voir resultats job bacteria-898db3c3 |
| BAGEL4 | Organism = *Lactiplantibacillus plantarum* |

**Phase 3**
| Etape | Valeur obtenue |
|-------|----------------|
| NCBI genomes | Taxon 1590, filtre = fermented milk |
| Roary | Core = 3321, Soft core = 0, Shell = 0, Cloud = 0, Total = 3321 |
| Phandango | 2 clades visibles, genes core = zone bleue homogene |